### Zadanie 4

1. Načítajte z knižnice `keras` dátovú množinu `fashion_mnist`, ktorá obsahuje obrázky oblečenia klasifikovaných do 10 typov.
2. Vhodne predspracujte dáta.
3. Navrhnite a naučte vhodnú architektúru s konvolúčnými vrstvami. Pri učení zobrazte priebeh presnosti na trénovacích a testovacích dátach.
4. Otestujte výslednú naučenú sieť na testovacej množine a vypočítajte výslednú presnosť.

In [2]:
import torchvision.transforms as transforms
import torch
import torch_directml
from torch import nn
from torch import optim
from torch.utils.data import DataLoader
from torchvision.datasets import FashionMNIST
from sklearn.model_selection import train_test_split
import wandb

from assignments.assignment4.dataloaders.custom_subset import CustomSubset
from assignments.assignment4.models.fashion_cnn_model import FashionCnnModel
from assignments.assignment4.utils.evaluation import Evaluation
from assignments.assignment4.utils.training import Training

LR = 0.001
BATCH_SIZE = 1024
EARLY_STOPPING = 5
EPOCHS = 100

transform = transforms.Compose([
    transforms.ToTensor()
])

dataset = FashionMNIST(".", download=True, transform=transform)
train_idx, test_idx, train_lab, test_lab = train_test_split(range(len(dataset)), dataset.targets, test_size=0.3, stratify=dataset.targets, random_state=42)
test_idx, val_idx = train_test_split(test_idx, test_size=1/3, stratify=test_lab, random_state=42)

classes = { idx: value for idx, value in enumerate(dataset.classes) }

trainset = CustomSubset(dataset, train_idx, classes)
testset = CustomSubset(dataset, test_idx, classes)
validset = CustomSubset(dataset, val_idx, classes)

trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)
testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False)
validloader = DataLoader(validset, batch_size=BATCH_SIZE, shuffle=False)

# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device = torch_directml.device()
model = FashionCnnModel()

loss = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, foreach=False)

wandb_config = dict(project="PMAD", entity="DP_Gereg", config={
    "learning rate": LR,
    "epochs": EPOCHS,
    "early stopping": EARLY_STOPPING,
    "loss calculator": str(loss),
    "batch_size": BATCH_SIZE,
    "model": str(model),
    "optimizer": str(optimizer),
})

wandb.login(key="a9f105e8b3bc98e07700e93201d4b02c1c75106d")

wandb.init(**wandb_config)

Training()(EPOCHS, device, optimizer, model, loss, trainloader, validloader, EARLY_STOPPING)
Evaluation()(loss, testloader, model, device)

if wandb.run is not None:
    wandb.finish()

Training run G5FWE1KAVHUMEXZ
Epoch 0 validation_report: {'balanced_accuracy': 0.10816666666666667, 'T-shirt/top': {'precision': 0.09615384615384616, 'recall': 0.008333333333333333, 'f1-score': 0.015337423312883436, 'support': 600.0}, 'Trouser': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 600.0}, 'Pullover': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 600.0}, 'Dress': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 600.0}, 'Coat': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 600.0}, 'Sandal': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 600.0}, 'Shirt': {'precision': 0.12554112554112554, 'recall': 0.09666666666666666, 'f1-score': 0.10922787193973635, 'support': 600.0}, 'Sneaker': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 600.0}, 'Bag': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 600.0}, 'Ankle boot': {'precision': 0.1068173532628509, 'recall': 0.9766666666666667, 'f1

C:\Users\lukiq\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\autograd\graph.py:744: UserWarning: The operator 'aten::native_dropout_backward' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 1 train_report: {'balanced_accuracy': 0.46814285714285714, 'T-shirt/top': {'precision': 0.3669135112477295, 'recall': 0.6252380952380953, 'f1-score': 0.4624460685040063, 'support': 4200.0}, 'Trouser': {'precision': 0.6524122807017544, 'recall': 0.5666666666666667, 'f1-score': 0.6065239551478083, 'support': 4200.0}, 'Pullover': {'precision': 0.33160813308687614, 'recall': 0.21357142857142858, 'f1-score': 0.25981173062997825, 'support': 4200.0}, 'Dress': {'precision': 0.5562526139690506, 'recall': 0.31666666666666665, 'f1-score': 0.4035806402670308, 'support': 4200.0}, 'Coat': {'precision': 0.23897879464285715, 'recall': 0.40785714285714286, 'f1-score': 0.3013722730471499, 'support': 4200.0}, 'Sandal': {'precision': 0.6120137863121615, 'recall': 0.5919047619047619, 'f1-score': 0.6017913338174776, 'support': 4200.0}, 'Shirt': {'precision': 0.23806571605703658, 'recall': 0.09142857142857143, 'f1-score': 0.1321176672974368, 'support': 4200.0}, 'Sneaker': {'precision': 0.73009639414494

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



evaluation_report: {'balanced_accuracy': 0.7749166666666667, 'T-shirt/top': {'precision': 0.7678714859437751, 'recall': 0.7966666666666666, 'f1-score': 0.7820040899795501, 'support': 1200.0}, 'Trouser': {'precision': 0.9783737024221453, 'recall': 0.9425, 'f1-score': 0.9601018675721562, 'support': 1200.0}, 'Pullover': {'precision': 0.5055653192735794, 'recall': 0.7191666666666666, 'f1-score': 0.5937392500859993, 'support': 1200.0}, 'Dress': {'precision': 0.6747868453105969, 'recall': 0.9233333333333333, 'f1-score': 0.7797325826882477, 'support': 1200.0}, 'Coat': {'precision': 0.6876332622601279, 'recall': 0.5375, 'f1-score': 0.6033676333021516, 'support': 1200.0}, 'Sandal': {'precision': 0.8860265417642467, 'recall': 0.9458333333333333, 'f1-score': 0.9149536477226925, 'support': 1200.0}, 'Shirt': {'precision': 0.4549266247379455, 'recall': 0.18083333333333335, 'f1-score': 0.2587954680977937, 'support': 1200.0}, 'Sneaker': {'precision': 0.8599033816425121, 'recall': 0.89, 'f1-score': 0.8